# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Natalie Ho and Evelyn Chen 

**ID**: nh424 

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/Downloads/hw5-natalie-evelyn5-Fall25`
   Installed OpenSSL ──────────── v1.6.0
   Installed GR_jll ───────────── v0.73.18+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed PlotUtils ────────── v1.4.4
   Installed Measures ─────────── v0.3.3
   Installed StaticArrays ─────── v1.9.15
   Installed MutableArithmetics ─ v1.6.7
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed Pango_jll ────────── v1.57.0+0
   Installed JSON ─────────────── v1.3.0
   Installed FFMPEG ───────────── v0.4.5
   Installed StaticArraysCore ─── v1.4.4
   Installed DataStructures ───── v0.19.3
   Installed Graphs ───────────── v1.13.1
   Installed METIS_jll ────────── v5.1.3+0
   Installed GR ───────────────── v0.73.18
   Installed StatsBase ────────── v0.34.8
   Installed GraphRecipes ─────── v0.5.15
   Installed ChainRulesCore ───── v1.26.0
   Installed HiGHS ────────────── v1.20.1
   Installed JuMP ─────────────── v1.29.3
   Installed Interpolations ───── v0.16.2
   Inst

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [7]:
#Problem 1.1 – overall recycling and ash fractions

# % of total mass for each component
mass = [15, 40, 5, 3, 2, 5, 18, 4, 2, 2, 1, 3]

# Combustion ash (%) for each component
ash = [8, 7, 5, 10, 15, 2, 2, 100, 100, 100, 100, 70]

# MRF recycling rate (%) for each component
recycle = [0, 55, 15, 10, 0, 30, 40, 60, 75, 80, 50, 0]

# Overall recycling fraction (R) and ash fraction (A)
R = sum(mass .* recycle) / 10000      # 37.75%  -> 0.3775
A = sum(mass .* ash) / 10000          # 16.41%  -> 0.1641

println("Overall recycling fraction R = ", round(R; digits=4))
println("Overall ash fraction A       = ", round(A; digits=4))

# If you also want the recycled and ash masses for each city:
city_total = [100.0, 90.0, 120.0]     # Mg/day for cities 1, 2, 3

recycled_mass = city_total .* R
ash_mass      = city_total .* A

println("\nRecycled mass by city (Mg/day): ", round.(recycled_mass; digits=2))
println("Ash mass by city (Mg/day):      ", round.(ash_mass; digits=2))


Overall recycling fraction R = 0.3775
Overall ash fraction A       = 0.1641

Recycled mass by city (Mg/day): [37.75, 33.98, 45.3]
Ash mass by city (Mg/day):      [16.41, 14.77, 19.69]


#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.



Here are the following decision variables for the municipal solid waste (MSW) management optimization model.

| Variable | Meaning |
|---------|---------|
| $W_{ij} \ge 0$ | Amount of waste (Mg/day) transported from city $i$ to disposal facility $j$, where $j \in \{\text{LF}, \text{MRF}, \text{WTE}\}$. These variables represent the primary flows of waste from the cities to each facility. |
| $R_{kj} \ge 0$ | Amount of residual waste or ash (Mg/day) transported from disposal facility $k$ to facility $j$. This includes non-recycled residue from the MRF sent to LF or WTE, and ash from the WTE sent to LF. These variables represent secondary flows after processing. |
| $Y_j \in \{0,1\}$ | Operational status of disposal facility $j$. $Y_j = 1$ if facility $j$ is open (fixed costs and capacity apply), and $Y_j = 0$ if the facility is closed and cannot receive waste. Facilities include LF, MRF, and WTE. |



#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).



The objective is to minimize the total daily cost of the MSW system, including:
1. Transportation costs  
2. Fixed operating costs for each facility  
3. Variable tipping and processing costs  

### Variables

- $I = \{1,2,3\}$ : set of cities  
- $J = \{\mathrm{LF}, \mathrm{MRF}, \mathrm{WTE}\}$ : set of facilities  
- $a = 1.5$ : transportation cost (\$/Mg·km)  
- $l_{ij}$ : distance from city $i$ to facility $j$  
- $l_{kj}$ : distance from facility $k$ to facility $j$  
- $c_j$ : fixed daily operating cost of facility $j$  
- $b_j$ : variable cost per Mg at facility $j$  

From the tables:
- $c_{\mathrm{LF}} = 2000$, $c_{\mathrm{MRF}} = 1500$, $c_{\mathrm{WTE}} = 2500$  
- $b_{\mathrm{LF}} = 50$, $b_{\mathrm{WTE}} = 60$  
- MRF effective variable cost:  
  $b_{\mathrm{MRF}} = 7 + 40R$ with $R = 0.3775$

The total mass handled at facility $j$ is  
$$H_j = \sum_{i\in I} W_{ij} + \sum_{k\in J} R_{kj}.$$


### Objective Function

The total transportation cost is

$$
C_{\text{transportation}} =
a \left(
\sum_{i\in I}\sum_{j\in J} l_{ij} W_{ij}
+
\sum_{k\in J}\sum_{j\in J} l_{kj} R_{kj}
\right).
$$

The total fixed facility cost is

$$
C_{\text{fixed}} = \sum_{j\in J} c_j Y_j.
$$

The total variable processing cost is

$$
C_{\text{var}} = \sum_{j\in J} b_j H_j.
$$

Putting everything together, the complete objective is

$$
\min_{W_{ij},\,R_{kj},\,Y_j}
\left[
a \left(
\sum_{i\in I}\sum_{j\in J} l_{ij} W_{ij}
+
\sum_{k\in J}\sum_{j\in J} l_{kj} R_{kj}
\right)
+
\sum_{j\in J} c_j Y_j
+
\sum_{j\in J} b_j H_j
\right].
$$

This matches the structure of the lecture formulation: transport costs, plus fixed facility costs, plus variable handling costs for each facility.

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.



Decision variables:

- $W_{ij}$ : waste sent from city $i$ to facility $j$ (Mg/day)  
- $R_{kj}$ : residual or ash sent from facility $k$ to facility $j$ (Mg/day)  
- $Y_j$ : binary variable indicating if facility $j$ is open (1) or closed (0)  

with cities $i \in I = \{1,2,3\}$ and facilities  
$j,k \in J = \{\mathrm{LF}, \mathrm{MRF}, \mathrm{WTE}\}$.

From Problem 1.1 we treat as **parameters**:

- overall recycling fraction $R = 0.3775$  
- overall ash fraction $A = 0.1641$  

Facility capacities (Mg/day):

- $K_{\mathrm{LF}} = 200$  
- $K_{\mathrm{MRF}} = 350$  
- $K_{\mathrm{WTE}} = 210$  

City waste generation (Mg/day):

- $S_1 = 100,\; S_2 = 90,\; S_3 = 120$.


### 1. City mass-balance constraints

All waste generated in each city must be sent to one or more facilities:

$$
\sum_{j\in J} W_{ij} = S_i, \qquad \forall i \in I.
$$

These equations ensure that no waste is left unassigned or “created” at a city.

### 2. Facility capacity constraints

Total incoming material to each facility (from cities and other facilities)
cannot exceed its capacity. We also link capacity to the binary variable
$Y_j$ so that if a facility is closed ($Y_j=0$) it cannot receive any waste:

$$
\sum_{i\in I} W_{ij}
+
\sum_{k\in J} R_{kj}
\;\le\;
K_j\, Y_j,
\qquad \forall j \in J.
$$

Here $K_j$ is the capacity of facility $j$.  
If $Y_j = 0$ the right-hand side is zero, forcing all flows to that facility
to be zero. If there is any positive flow, it will be optimal to choose
$Y_j = 1$ and incur the fixed cost in the objective.

### 3. MRF mass balance (recycling and residue)

Let the total mass entering the MRF be

$$
H_{\mathrm{MRF}} = \sum_{i\in I} W_{i,\mathrm{MRF}}.
$$

A fraction $R$ of this flow is recycled and leaves the system, while the
remaining fraction $(1-R)$ becomes residue. All residue must go either to
LF or to WTE:

$$
R_{\mathrm{MRF},\mathrm{LF}} + R_{\mathrm{MRF},\mathrm{WTE}}
=
(1 - R)\, H_{\mathrm{MRF}}
=
(1 - R)\, \sum_{i\in I} W_{i,\mathrm{MRF}}.
$$

This constraint enforces conservation of mass at the MRF: all non-recycled
waste is accounted for in the residual flows.


### 4. WTE mass balance and ash generation

Let the total mass entering the WTE be

$$
H_{\mathrm{WTE}} =
\sum_{i\in I} W_{i,\mathrm{WTE}} + R_{\mathrm{MRF},\mathrm{WTE}}.
$$

A fraction $A$ of this incoming mass becomes bottom ash that must be sent to
the landfill. We represent this with the residual variable from WTE to LF:

$$
R_{\mathrm{WTE},\mathrm{LF}} = A\, H_{\mathrm{WTE}}
=
A\left(
\sum_{i\in I} W_{i,\mathrm{WTE}} + R_{\mathrm{MRF},\mathrm{WTE}}
\right).
$$

This constraint ensures that the ash produced by the WTE is consistent with
the ash fraction computed in Problem 1.1 and that all ash is routed to the
landfill.


### 5. Domain constraints

All flow variables must be nonnegative:

$$
W_{ij} \ge 0, \qquad \forall i \in I,\; j \in J,
$$

$$
R_{kj} \ge 0, \qquad \forall k \in J,\; j \in J.
$$

Facility status variables are binary:

$$
Y_j \in \{0,1\}, \qquad \forall j \in J.
$$

In practice, we only define $R_{kj}$ for physically meaningful connections
(e.g., MRF $\to$ WTE or LF, WTE $\to$ LF); all other $R_{kj}$ are either
omitted from the model or constrained to zero.


Together, these constraints enforce (i) conservation of mass at cities and
facilities, (ii) consistency of recycling and ash generation with the
fractions computed in Problem 1.1, and (iii) capacity and operational limits
for each disposal option.

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.